메타클래스의 다른 유용한 기능으로, 클래스가 정의된 후 사용되기 이전 시점에 프로퍼티를 변경하거나 표시할 수 있다.

In [ ]:
class Field:
    def __init__(self, name):
        self.name = name
        self.internal_name = '_' + self.name

    def __get__(self, instance, instance_type):
        if instance is None:
            return self
        return getattr(instance, self.internal_name, '')

    def __set__(self, instance, value):
        setattr(instance, self.internal_name, value)

class Customer:
    # 클래스 애트리뷰트
    first_name = Field('first_name')
    last_name = Field('last_name')
    prefix = Field('prefix')
    suffix = Field('suffix')

cust = Customer()
print(f'이전: {cust.first_name!r} {cust.__dict__}')
cust.first_name = '유클리드'
print(f'이후: {cust.first_name!r} {cust.__dict__}')

위 방식은 `first_name = Field('first_name')` 같은 불필요한 중복이 있다. 메타클래스를 이용하면 이런 문제를 해결할 수 있다.

In [ ]:
class Meta(type):
    def __new__(meta, name, bases, class_dict):
        for key, value in class_dict.items():
            if isinstance(value, Field):
                value.name = key
                value.internal_name = '_' + key
        cls = type.__new__(meta, name, bases, class_dict)
        return cls

class DatabaseRow(metaclass=Meta):
    pass


class Field:
    def __init__(self):
        # 이 두 정보를 메타클래스가 채워 준다
        self.name = None
        self.internal_name = None

    def __get__(self, instance, instance_type):
        if instance is None:
            return self
        return getattr(instance, self.internal_name, '')

    def __set__(self, instance, value):
        setattr(instance, self.internal_name, value)

class BetterCustomer(DatabaseRow):
    first_name = Field()
    last_name = Field()
    prefix = Field()
    suffix = Field()

cust = BetterCustomer()
print(f'이전: {cust.first_name!r} {cust.__dict__}')
cust.first_name = '오일러'
print(f'이후: {cust.first_name!r} {cust.__dict__}')

위 방식의 문제점은 `BetterCustomer`가 `DatabaseRow`를 상속해야 한다는 점이다. 상속하는걸 잊어버리거나 클래스 계층구조로 인해 상속을 할 수 없는 경우도 있다.

In [ ]:
class BrokenCustomer:
    first_name = Field()
    last_name = Field()
    prefix = Field()
    suffix = Field()

cust = BrokenCustomer()
cust.first_name = '메르센'

이러한 문제를 해결하는 방법으로 `__set_name__` 특별 메소드(파이썬 3.6 이후)를 사용할 수 있다. 메타클래스 정의를 아예 피하고, `Meta.__new__`가 하던 일을 `__set_name__`에서 처리한다.

이젠 특정 기반 클래스를 상속하거나 메타클래스를 사용하지 않아도 `Field` 디스크립터가 제공하는 기능을 모두 활용할 수 있다.

In [ ]:
class Field:
    def __init__(self):
        self.name = None
        self.internal_name = None

    def __set_name__(self, owner, name):
        # 클래스가 생성될 때 모든 스크립터에 대해 이 메서드가 호출된다
        self.name = name
        self.internal_name = '_' + name

    def __get__(self, instance, instance_type):
        if instance is None:
            return self
        return getattr(instance, self.internal_name, '')

    def __set__(self, instance, value):
        setattr(instance, self.internal_name, value)


class FixedCustomer:
    first_name = Field()
    last_name = Field()
    prefix = Field()
    suffix = Field()

cust = FixedCustomer()
print(f'이전: {cust.first_name!r} {cust.__dict__}')
cust.first_name = '메르센'
print(f'이후: {cust.first_name!r} {cust.__dict__}')